# Portfolio: drift, covariance and attribution

Run from an environment with the project installed after migrations and real market-data import. Load real provider observations before executing this notebook.


In [ ]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir('..')
from sqlalchemy import select

from app.db.database import SessionLocal
from app.db.models import Dataset
from app.research.returns import simple_returns
from app.services.research import price_panel

db=SessionLocal()
dataset=db.scalar(select(Dataset).where(~Dataset.source.ilike("%SYNTHETIC%")).order_by(Dataset.created_at.desc()))
assert dataset is not None, 'Run migrations and ingest_market_data first'
panel=price_panel(db,dataset.id,['AAPL','MSFT','SPY'])
returns=simple_returns(panel)
print('Source:',dataset.source,'Dataset:',dataset.id)

In [ ]:
from app.research.attribution import linked_contribution
from app.research.portfolio import portfolio
from app.research.risk import covariance_risk

weights={'AAPL':.5,'MSFT':.5}
sim=portfolio(panel[['AAPL','MSFT']],weights,cost_bps=10)
print(covariance_risk(returns[['AAPL','MSFT']],[.5,.5]))
print(linked_contribution(sim))
sim['path'].wealth.plot(title='Monthly portfolio')

## Interpretation

Reconcile contribution totals to terminal wealth. Explain why target-weight covariance risk differs from realized path risk.


In [ ]:
db.close()